# Example 7.8.4 / 1: Request a Fixed-MIP Postsolve

Companion notebook for **7.8.4 Fixed-MIP Postsolve and Dual Values** in the AMPL lab Documentation, section "Request a Fixed-MIP Postsolve". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Request a Fixed-MIP Postsolve

The option `mip:basis=1` (`0`, the default, skips it) asks the driver to solve the fixed model
after the MIP and return its
basis status (`.sstatus`) and dual values (`.dual`); `alg:sens=1` additionally requests
sensitivity ranges where supported. The fixed-charge steel model from
Binary Variables and Logical Conditions (AMPL lab documentation, section 7.2) shows the difference:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products
param rate {PROD} > 0;            # tons produced per hour
param profit {PROD};              # profit per ton
param market {PROD} >= 0;         # most that can be sold, in tons
param setup {PROD} >= 0;          # fixed cost paid if the product is made
param avail >= 0;                 # hours available

var Make {p in PROD} >= 0, <= market[p];  # tons produced
var Use {PROD} binary;                     # 1 if production of the product is active

maximize NetProfit:
    sum {p in PROD} (profit[p] * Make[p] - setup[p] * Use[p]);
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;  # hours available
subject to Activate {p in PROD}: Make[p] <= market[p] * Use[p];      # production only if set up

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["setup"] = {"bands": 20000, "coils": 10000, "plate": 50000}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.option["solver_msg"] = 0
for options in ["", "mip:basis=1"]:
    ampl.solve(solver="gurobi", gurobi_options=options, verbose=False)
    print(options or "defaults", "|", ampl.solve_result, ampl.obj["NetProfit"].value(),
          "| Time.dual", ampl.con["Time"].dual(), "| Time.sstatus", ampl.get_value("Time.sstatus"))
print("Activate duals:", ampl.con["Activate"].get_values().to_dict())

defaults | solved 162000.0 | Time.dual 0.0 | Time.sstatus none
mip:basis=1 | solved 162000.0 | Time.dual 4200.0 | Time.sstatus upp
Activate duals: {'bands': 0, 'coils': 0, 'plate': 2.75}


### Pure AMPL: data.dat and model.run

`ampl.con["Time"].dual()` reads the dual value and `ampl.get_value("Time.sstatus")` the basis
status. The plain MIP solve returns no basis (`none`) and a dual of 0. With `mip:basis=1`, the
fixed model has bands at their market limit and coils using the last 10 hours, so one more hour
is worth 140 tons of coils at 30, a dual of 4,200.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: PROD: rate profit market setup :=
  bands  200  25  6000  20000
  coils  140  30  4000  10000
  plate  160  29  3500  50000 ;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver gurobi;
option solver_msg 0;
option gurobi_options 'mip:basis=1';
solve;
display Time.dual, Time.sstatus, Activate.dual;

Gurobi 13.0.0:   mip:basis = 1
Time.dual = 4200
Time.sstatus = upp

Activate.dual [*] :=
bands  0
coils  0
plate  2.75
;

